### Lặp Gauss-Seidel

### Yêu cầu trình bày (Ghi vào bài thi)
**1. Điều kiện hội tụ:**
Ma trận $A$ chéo trội hàng:
$$|A_{ii}| > \sum_{j \neq i} |A_{ij}| \quad \forall i$$

**2. Công thức lặp Gauss-Seidel:**
$$x_i^{(k+1)} = \frac{1}{A_{ii}} \left( b_i - \sum_{j=1}^{i-1} A_{ij} x_j^{(k+1)} - \sum_{j=i+1}^n A_{ij} x_j^{(k)} \right)$$
*(Nghĩa là: nghiệm $x_i$ ở bước mới nhất được tính bằng cách dùng ngay các $x_j$ mới nhất vừa tính được ở cùng bước)*


In [13]:
import numpy as np
from IPython.display import Markdown, display

def Gauss_Seidel_Ax_B(A_input, b_input, max_iter=None, epsilon=None, x0=None):
    def matrix_to_latex(M, precision=4):
        if not isinstance(M, np.ndarray): return str(M)
        elif M.ndim == 1:
            inner = " \\\\ ".join([f"{x:.{precision}f}" for x in M])
            return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}"
        else:
            rows = " \\\\ ".join([" & ".join([f"{x:.{precision}f}" for x in row]) for row in M])
            return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"

    A = np.array(A_input, dtype=float)
    b = np.array(b_input, dtype=float).flatten()
    n = len(b)
    
    if x0 is None:
        X_k = np.zeros(n)
    else:
        X_k = np.array(x0, dtype=float)
        
    md = []
    md.append("## ❖ PHƯƠNG PHÁP LẶP GAUSS-SEIDEL (DẠNG $Ax = b$)")
    md.append("---\n")
    md.append("> **📝 HƯỚNG DẪN TRÌNH BÀY BÀI THI:**")
    md.append("> - **Bước 1:** Đưa hệ về dạng $x = Bx + d$ bằng cách rút $x_i$ từ phương trình thứ $i$. Tính $||B||_\\infty$.")
    md.append("> - **Bước 2:** Viết công thức lặp Gauss-Seidel dạng khai triển.")
    md.append("> - **Bước 3:** Kẻ bảng quá trình lặp và ghi số liệu.")
    md.append("> - **Bước 4:** Kết luận nghiệm và sai số nếu được yêu cầu.")
    
    # Biến đổi Ax=b thành x=Bx+d
    D = np.diag(np.diag(A))
    L = np.tril(A, -1)
    U = np.triu(A, 1)
    
    # B = -D^-1 (L + U)
    D_inv = np.linalg.inv(D)
    B = -D_inv @ (L + U)
    d = D_inv @ b
    
    md.append("\n### I. RÚT X VÀ KIỂM TRA ĐIỀU KIỆN HỘI TỤ")
    md.append(f"Hệ tương đương với dạng $x = Bx + d$, với:")
    md.append(f"$$ B = {matrix_to_latex(B, precision=4)} $$")
    md.append(f"$$ d = {matrix_to_latex(d, precision=4)} $$")
    
    norm_B = np.linalg.norm(B, np.inf)
    md.append(f"Chuẩn vô cùng của ma trận $B$: $||B||_\\infty = {norm_B:.4f}$")
    if norm_B < 1:
        md.append(f"Vì $||B||_\\infty < 1$, phương pháp lặp chắc chắn hội tụ.")
    else:
        md.append(f"**Cảnh báo:** $||B||_\\infty \\ge 1$, phương pháp chưa chắc hội tụ.")

    md.append("\n### II. CÔNG THỨC LẶP GAUSS-SEIDEL (KHAI TRIỂN)")
    
    for i in range(n):
        terms = []
        for j in range(n):
            val = B[i, j]
            if abs(val) < 1e-10: continue
            
            sign = " + " if val > 0 else " - "
            if j < i:
                terms.append(f"{sign}{abs(val):.4f} x_{{{j+1}}}^{{(k+1)}}")
            else:
                terms.append(f"{sign}{abs(val):.4f} x_{{{j+1}}}^{{(k)}}")
        
        formula = "".join(terms).lstrip(" + ")
        d_sign = f" + {d[i]:.4f}" if d[i] >= 0 else f" - {abs(d[i]):.4f}"
        md.append(f"- $x_{{{i+1}}}^{{(k+1)}} = {formula}{d_sign}$")
        
    md.append("\n---\n### III. BẢNG QUÁ TRÌNH LẶP")
    
    header = "| $k$ | " + " | ".join([f"$x_{{{i+1}}}$" for i in range(n)]) + " | Sai số $||x^{(k)} - x^{(k-1)}||_\\infty$ |"
    md.append(header)
    md.append("|---" * (n + 2) + "|")
    
    row0 = f"| 0 | " + " | ".join([f"{x:.5f}" for x in X_k]) + " | - |"
    md.append(row0)
    
    k = 1
    max_safe_iters = max_iter if max_iter is not None else 100
    
    while True:
        X_new = np.copy(X_k)
        for i in range(n):
            s1 = sum(B[i][j] * X_new[j] for j in range(i))
            s2 = sum(B[i][j] * X_k[j] for j in range(i, n))
            X_new[i] = s1 + s2 + d[i]
            
        sai_so = np.linalg.norm(X_new - X_k, np.inf)
        row = f"| {k} | " + " | ".join([f"{x:.5f}" for x in X_new]) + f" | {sai_so:.5f} |"
        md.append(row)
        X_k = X_new
        
        stop_eps = (epsilon is not None) and (sai_so <= epsilon)
        stop_iter = (max_iter is not None) and (k >= max_iter)
        if stop_eps or stop_iter or k >= max_safe_iters:
            break
        k += 1
        
    md.append("\n---\n### IV. KẾT LUẬN")
    md.append(f"Hệ dừng lại tại bước lặp $k = {k}$. Nghiệm gần đúng là:")
    md.append(f"$$ x^{{({k})}} = {matrix_to_latex(X_k, precision=5)} $$")
    
    display(Markdown('\n'.join(md)))

# Ma trận cung cầu A (Kích thước 10x10)

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
A = np.array([
    [10.0, 1.0,  2.0],
    [ 1.0, 10.0, 3.0],
    [ 2.0,  3.0, 10.0]
], dtype=float)

b = np.array([7.0, 8.0, 9.0], dtype=float)
# ═══════════════════════════════════════════════════════════════════

Gauss_Seidel_Ax_B(A, b, max_iter=None, epsilon=1e-6)


## ❖ PHƯƠNG PHÁP LẶP GAUSS-SEIDEL (DẠNG $Ax = b$)
---

> **📝 HƯỚNG DẪN TRÌNH BÀY BÀI THI:**
> - **Bước 1:** Đưa hệ về dạng $x = Bx + d$ bằng cách rút $x_i$ từ phương trình thứ $i$. Tính $||B||_\infty$.
> - **Bước 2:** Viết công thức lặp Gauss-Seidel dạng khai triển.
> - **Bước 3:** Kẻ bảng quá trình lặp và ghi số liệu.
> - **Bước 4:** Kết luận nghiệm và sai số nếu được yêu cầu.

### I. RÚT X VÀ KIỂM TRA ĐIỀU KIỆN HỘI TỤ
Hệ tương đương với dạng $x = Bx + d$, với:
$$ B = \begin{bmatrix} 0.0000 & -0.1000 & -0.2000 \\ -0.1000 & 0.0000 & -0.3000 \\ -0.2000 & -0.3000 & 0.0000 \end{bmatrix} $$
$$ d = \begin{bmatrix} 0.7000 \\ 0.8000 \\ 0.9000 \end{bmatrix} $$
Chuẩn vô cùng của ma trận $B$: $||B||_\infty = 0.5000$
Vì $||B||_\infty < 1$, phương pháp lặp chắc chắn hội tụ.

### II. CÔNG THỨC LẶP GAUSS-SEIDEL (KHAI TRIỂN)
- $x_{1}^{(k+1)} = - 0.1000 x_{2}^{(k)} - 0.2000 x_{3}^{(k)} + 0.7000$
- $x_{2}^{(k+1)} = - 0.1000 x_{1}^{(k+1)} - 0.3000 x_{3}^{(k)} + 0.8000$
- $x_{3}^{(k+1)} = - 0.2000 x_{1}^{(k+1)} - 0.3000 x_{2}^{(k+1)} + 0.9000$

---
### III. BẢNG QUÁ TRÌNH LẶP
| $k$ | $x_{1}$ | $x_{2}$ | $x_{3}$ | Sai số $||x^{(k)} - x^{(k-1)}||_\infty$ |
|---|---|---|---|---|
| 0 | 0.00000 | 0.00000 | 0.00000 | - |
| 1 | 0.70000 | 0.73000 | 0.54100 | 0.73000 |
| 2 | 0.51880 | 0.58582 | 0.62049 | 0.18120 |
| 3 | 0.51732 | 0.56212 | 0.62790 | 0.02370 |
| 4 | 0.51821 | 0.55981 | 0.62842 | 0.00231 |
| 5 | 0.51834 | 0.55964 | 0.62844 | 0.00017 |
| 6 | 0.51835 | 0.55963 | 0.62844 | 0.00001 |
| 7 | 0.51835 | 0.55963 | 0.62844 | 0.00000 |

---
### IV. KẾT LUẬN
Hệ dừng lại tại bước lặp $k = 7$. Nghiệm gần đúng là:
$$ x^{(7)} = \begin{bmatrix} 0.51835 \\ 0.55963 \\ 0.62844 \end{bmatrix} $$